In [4]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from time import perf_counter

from rkhs_epsilon import RKHSEpsilonMachine
from toy_datasets import two_regime_gauss

rng = np.random.default_rng(2137)

In [5]:
series, states_true, T_true, pi_true = two_regime_gauss(seed=2137)

1. Shuffle surrogate - randomly permute the series, destroying all temporal dependence
2. Phase randomized surrogate - preserve the linear autocorrelation structure (amplitude spectrum) while destroying nonlinear dependencies by randomizing Fourier phases.
3. Block shuffle surrogate - split the series into non-overlapping blocks and randomly permutes their order; preserves local volatility clustering within blocks while disrupting longer-range sequential organization
4. GARCH surrogate - preserve conditional heteroskedasticity but replaces any richer causal structure with a memoryless innovation process.

Draw a uniform random permutation $\sigma \in S_N$ and return $\tilde{x}_t = x_{\sigma(t)}$

In [3]:
def shuffle_surrogate(series: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    out = series.copy()
    rng.shuffle(out)
    return out

Take the DFT, keep the amplitudes $|\hat{X}_k|$, replace the phases with i.i.d. draws $\phi_k \sim \mathcal{U}[0, 2\pi)$, then invert:

$$\tilde{X}_k = |\hat{X}_k|\, e^{i\phi_k}, \qquad \tilde{x} = \mathcal{F}^{-1}(\tilde{X})$$

DC ($k=0$) and, for even $N$, the Nyquist component are pinned to phase $0$ to guarantee a real-valued output. This preserves the linear autocorrelation structure (same power spectrum) while destroying any nonlinear dependence.

In [ ]:
def phase_randomize_surrogate(series: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    n = len(series)
    fft_vals = np.fft.rfft(series)
    amplitudes = np.abs(fft_vals)

    # Random phases; keep DC and (if n is even) Nyquist real
    n_freqs = len(fft_vals)
    phases = rng.uniform(0, 2 * np.pi, n_freqs)
    phases[0] = 0.0
    if n % 2 == 0:
        phases[-1] = 0.0

    new_fft = amplitudes * np.exp(1j * phases)
    surrogate = np.fft.irfft(new_fft, n=n)
    return surrogate.astype(series.dtype)

Chop the series into non-overlapping blocks of size $b \approx \sqrt{N}$, then permute the block order:

$$\tilde{x} = \bigl[B_{\sigma(1)},\, B_{\sigma(2)},\, \ldots,\, B_{\sigma(m)}\bigr]$$

Within each block, the original dynamics are intact - so local volatility clustering and short-range autocorrelation survive. Long-range sequential organisation is destroyed.

In [ ]:
def block_shuffle_surrogate(series: np.ndarray, rng: np.random.Generator, block_size: int | None = None) -> np.ndarray:
    n = len(series)
    if block_size is None:
        block_size = max(5, int(np.sqrt(n)))

    n_blocks = n // block_size
    if n_blocks < 2:
        # Fall back to simple shuffle if series is too short
        return shuffle_surrogate(series, rng)

    blocks = [series[i * block_size: (i + 1) * block_size] for i in range(n_blocks)]
    rng.shuffle(blocks)

    # Append any leftover tail
    tail = series[n_blocks * block_size:]
    return np.concatenate(blocks + ([tail] if len(tail) else []))

The model: $r_t = \sigma_t z_t$ with $z_t \overset{iid}{\sim} \mathcal{N}(0,1)$ and $\sigma_t^2 = \omega + \alpha\, r_{t-1}^2 + \beta\, \sigma_{t-1}^2$

Parameters are estimated from the input series via moment matching: the unconditional variance pins $\omega = \hat{\sigma}^2(1 - \alpha - \beta)$, and the persistence $\alpha + \beta$ is read off from $\text{ACF}_1(r^2)$. A fresh draw of innovations $z_t$ then replaces the original dynamics with a memoryless GARCH process that has the same volatility envelope but no richer causal structure.

In [ ]:
def garch_surrogate(
    series: np.ndarray,
    rng: np.random.Generator,
) -> np.ndarray:
    n = len(series)
    r = series - series.mean()
    r2 = r ** 2

    # Moment-based parameter estimation
    unconditional_var = r2.mean()

    # Persistence α+β ≈ ACF(r², lag=1)
    acf1 = float(np.corrcoef(r2[:-1], r2[1:])[0, 1])
    ab = np.clip(acf1, 0.05, 0.97)
    alpha = min(0.15, ab * 0.2)
    beta = ab - alpha
    omega = max(unconditional_var * (1.0 - alpha - beta), 1e-10)

    # Simulate σ²_t = ω + α·r²_{t-1} + β·σ²_{t-1}
    sigma2 = np.empty(n)
    out = np.empty(n)
    sigma2[0] = unconditional_var
    z = rng.standard_normal(n)
    out[0] = np.sqrt(sigma2[0]) * z[0]

    for t in range(1, n):
        sigma2[t] = omega + alpha * out[t - 1] ** 2 + beta * sigma2[t - 1]
        out[t] = np.sqrt(sigma2[t]) * z[t]

    return out.astype(series.dtype)